En este notebook se presenta el calculo del valor de inflacion de varianza de forma manual y se compara con los resultados generados por la función de la biblioteca statsmodels

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

from statsmodels.stats.outliers_influence import variance_inflation_factor

usamos el dataset de publicidad

In [2]:
df = pd.read_csv("publicidad.csv")

In [3]:
df.drop(columns="id",inplace=True)

In [4]:
X=df.drop(columns=["sales"])
Y=df["sales"]

recordemos que los valores deben ser menores a 5 para no ver evidencia de multicolinealidad

In [6]:

VIF = pd.DataFrame()
VIF["Covariables"] = X.columns
VIF["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
VIF

,Covariables,VIF
0,TV,2.486772
1,radio,3.285462
2,newspaper,3.055245


hacemos el calculo manual del vif ajustando un modelo de regresión iterando para cada una de las variables predictoras y dejando por fuera la variable objetivo. Calculamos el vif con la formula (1/1-r2)

In [6]:
for i in X.columns:
    model_var_n = LinearRegression().fit(X.drop(columns=i),X[i])
    r2_model_var_n = model_var_n.score(X.drop(columns=i),X[i])
    vif_model_var_n = 1/(1-r2_model_var_n)
    vif_model_var_n
    print("column=",i," r2: ",r2_model_var_n," vif ",vif_model_var_n)

column= TV  r2:  0.004589623174239721  vif  1.0046107849396502
column= radio  r2:  0.1266008772420567  vif  1.1449519171055353
column= newspaper  r2:  0.12678045656223502  vif  1.1451873787239288


vemos que el calculo manual da resultados distintos a los de la funcion de statsmodels. Veamos que sucede si agregamos la constante del intercepto a nuestro df y volvemos a calcular con la función

In [7]:
aux_x = X.copy()
aux_x["constant"]=1
VIF = pd.DataFrame()
VIF["Covariables"] = aux_x.columns
VIF["VIF"] = [variance_inflation_factor(aux_x.values, i) for i in range(aux_x.shape[1])]
VIF.head(3)

,Covariables,VIF
0,TV,1.004611
1,radio,1.144952
2,newspaper,1.145187


en este caso los valores coinciden. Lo cual significa que la funcion de statsmodels no incluye el intercepto en los modelos que ajusta entre las covariables 